In [10]:
from qdrant_client import QdrantClient

client = QdrantClient(host="localhost", port=6333)

collection_name = "annual_report_debug"

try:
    client.delete_collection(collection_name)
    print("Old collection deleted")
except:
    print("Collection did not exist")

Old collection deleted


In [11]:
import os
from dotenv import load_dotenv

load_dotenv()

print("GROQ KEY loaded:", os.getenv("GROQ_API_KEY")[:10])

GROQ KEY loaded: gsk_YRzXDf


In [12]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "../data/annual_reports/hsbc_annual_report.pdf"

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

print("Total pages:", len(docs))

print("\nExample page text:\n")
print(docs[10].page_content[:500])

Total pages: 168

Example page text:

Other operating income of £158m increased by £52m compared 
with 2024. The prior year included foreign exchange translation losses 
of £54m associated with the sale of our businesses in Armenia and 
Russia. This was offset by £64m of foreign currency translation 
reserve losses recognised on completion of the disposal of our 
French life insurance business in 2025. 
Expected Credit Losses ('ECL') of £154m in 2025 was £9m lower 
compared with 2024. In both years ECL was primarily comprised of 
st


In [13]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(docs)

print("Total chunks:", len(chunks))

print("\nExample chunk:\n")
print(chunks[100].page_content[:400])

Total chunks: 1355

Example chunk:

due to the reclassification to held-for-sale of the UK life Insurance 
business.
Liabilities of disposal groups held for sale decreased by £7.4bn or 
32.0% . In 2024, this included the reclassification of the France Life 
Insurance business (£18.7bn) and the private banking business in 
Germany (£1.8bn). In 2025, the UK life insurance business and the 
custody business in Germany were reclassified


In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

test_vector = embeddings.embed_query("HSBC revenue")

print("Embedding dimension:", len(test_vector))

Loading weights: 100%|█████████████| 103/103 [00:00<00:00, 496.85it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384


In [15]:
from qdrant_client import QdrantClient

client = QdrantClient(
    host="localhost",
    port=6333
)

print("Connected to Qdrant")

Connected to Qdrant


In [16]:
from langchain_community.vectorstores import Qdrant

collection_name = "annual_report_debug"

vectorstore = Qdrant.from_documents(
    chunks,
    embeddings,
    url="http://localhost:6333",
    collection_name=collection_name
)

print("Embeddings uploaded")

Embeddings uploaded


In [17]:
query = "What risks does  HSBC mention?"

docs = vectorstore.similarity_search(query, k=5)

for i, doc in enumerate(docs):
    print("\nRESULT", i+1)
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:400])
    print("-"*50)


RESULT 1
Page: 34
risks;
– broader and more prolonged conflict in the Middle East and the 
Russia-Ukraine war, which undermine confidence and investment; 
and
– continued differences between the US and China, which affect 
economic confidence and the global goods trade and supply chains 
for critical technologies.
Risk review
HSBC Bank plc Annual Report and Accounts 2025
33
--------------------------------------------------

RESULT 2
Page: 14
disruptions during a period of heightened execution risk, driven by the complexity and scale of ongoing strategic, regulatory and 
technological change.
Risk Description
Strategic Report | Risk overview
HSBC Bank plc Annual Report and Accounts 2025
13
--------------------------------------------------

RESULT 3
Page: 17
replaced by alternative US tariffs. The risk of higher sector-based 
tariffs, alongside reduced global trade volumes and supply chain 
disruptions, continues. The potential for broader escalation of tariffs or 
a trade war is also

In [18]:
docs = vectorstore.max_marginal_relevance_search(
    query,
    k=6,
    fetch_k=20
)

for i, doc in enumerate(docs):
    print("\nRESULT", i+1)
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:400])


RESULT 1
Page: 34
risks;
– broader and more prolonged conflict in the Middle East and the 
Russia-Ukraine war, which undermine confidence and investment; 
and
– continued differences between the US and China, which affect 
economic confidence and the global goods trade and supply chains 
for critical technologies.
Risk review
HSBC Bank plc Annual Report and Accounts 2025
33

RESULT 2
Page: 69
Crime, while the HSBC Bank plc Risk Management Meeting retains 
oversight of matters relating to financial crime.
Key risk management processes 
We will not tolerate knowingly conducting business with individuals or 
entities believed to be engaged in criminal activity. We require 
everybody in HSBC to play their role in maintaining effective systems 
and controls to prevent and detect financial c

RESULT 3
Page: 29
–  personal — 1 — — 1 — — — — — — — — — —
–  corporate and 
commercial 558 2 7 — 567 — — (2) — (2) — — 28.6 — 0.4
–  financial 553 21 1 — 575 — — (1) — (1) — — 100.0 — 0.2
At 31 Dec 2

In [19]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

In [20]:
context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are a financial analyst.

Use the HSBC annual report context to answer.

Context:
{context}

Question:
{query}

Answer:
"""

In [21]:
response = llm.invoke(prompt)

print(response.content)

Based on the provided context from the HSBC annual report, the following risks are mentioned:

1. **Geopolitical risks**: 
   - Broader and more prolonged conflict in the Middle East.
   - The Russia-Ukraine war, which could undermine confidence and investment.
   - Continued differences between the US and China, affecting economic confidence, global goods trade, and supply chains for critical technologies.

2. **Financial Crime Risk**: 
   - The risk of conducting business with individuals or entities engaged in criminal activity.
   - The need to maintain effective systems and controls to prevent and detect financial crime.

3. **Regulatory and Legal Risks**:
   - Other regulatory investigations, reviews, and litigation, including enquiries, examinations, requests for information, and legal proceedings by tax authorities, regulators, competition, and law enforcement authorities.

4. **Climate Risk**:
   - The potential impact of climate risk on HSBC’s risk taxonomy, including how it 

In [13]:
questions = [
    "What are HSBC's strategic priorities?",
    "What risks does HSBC mention?",
    "What was HSBC revenue?",
    "How did HSBC perform in Asia?"
]

for q in questions:
    docs = vectorstore.max_marginal_relevance_search(q, k=5, fetch_k=20)

    print("\nQUESTION:", q)
    print("TOP PAGE:", docs[0].metadata.get("page"))


QUESTION: What are HSBC's strategic priorities?
TOP PAGE: 20

QUESTION: What risks does HSBC mention?
TOP PAGE: 34

QUESTION: What was HSBC revenue?
TOP PAGE: 126

QUESTION: How did HSBC perform in Asia?
TOP PAGE: 4
